# 01. 탐색적 데이터 분석 (EDA)
## Credit Card Fraud Detection
---
**데이터**: Kaggle Credit Card Fraud Dataset (2013년 유럽 카드사 실거래)  
**목표**: 284,807건 거래에서 492건(0.17%) 사기 탐지  
**키 챌린지**: 극단적 클래스 불균형, PCA 익명화 피처

> 실제 데이터: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
> `data/creditcard.csv` 에 저장 후 실행하세요.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from src.utils import load_config, set_seed, get_logger
from src.data_loader import load_data, describe_data
from src.visualize import plot_class_distribution, plot_amount_distribution

cfg = load_config('../configs/config.yaml')
set_seed(cfg['project']['seed'])
logger = get_logger('EDA')

print("설정 로드 완료")
print(f"프로젝트: {cfg['project']['name']} v{cfg['project']['version']}")

In [ ]:
# ── 데이터 로드 ─────────────────────────────────────────────
df = load_data(cfg)
print(f"\n데이터 크기: {df.shape}")
df.head()

In [ ]:
# ── 기초 통계 ───────────────────────────────────────────────
stats = describe_data(df)
print(f"총 거래:   {stats['total']:,}건")
print(f"사기:      {stats['fraud_count']:,}건  ({stats['fraud_rate']:.4%})")
print(f"정상:      {stats['normal_count']:,}건")
print(f"\n거래 금액:")
print(f"  정상 평균:  ${stats['amount_stats']['normal_mean']:.2f}")
print(f"  사기 평균:  ${stats['amount_stats']['fraud_mean']:.2f}")
print(f"  정상 중앙값: ${stats['amount_stats']['normal_median']:.2f}")
print(f"  사기 중앙값: ${stats['amount_stats']['fraud_median']:.2f}")
print(f"\n결측치: {sum(stats['null_counts'].values())}개")
print(f"중복 행: {stats['duplicates']:,}개")

In [ ]:
# ── 클래스 불균형 시각화 ────────────────────────────────────
fig = plot_class_distribution(df['Class'], save_path='../outputs/figures/01_class_dist.png')
plt.show()
print("\n이 불균형 수준에서 단순 정확도는 의미없음")
print(f"   항상 '정상'으로 예측해도 정확도 = {1 - stats['fraud_rate']:.4%}")

In [ ]:
# ── 거래 금액 분포 ──────────────────────────────────────────
fig = plot_amount_distribution(df, save_path='../outputs/figures/02_amount_dist.png')
plt.show()

In [ ]:
# ── V 피처 분포 비교 (정상 vs 사기) ───────────────────────
# 실제 데이터에서 사기와 가장 큰 차이를 보이는 피처 식별
fraud  = df[df['Class'] == 1]
normal = df[df['Class'] == 0].sample(2000, random_state=42)  # 시각화용 다운샘플

# 각 V 피처의 정상/사기 평균 차이 절대값
v_cols = [f'V{i}' for i in range(1, 29)]
diff = (fraud[v_cols].mean() - normal[v_cols].mean()).abs().sort_values(ascending=False)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Top 10 Discriminative Features (정상 vs 사기)', fontsize=13, color='#e6edf3')

for ax, col in zip(axes.flat, diff.head(10).index):
    ax.set_facecolor('#161b22')
    ax.hist(normal[col], bins=50, alpha=0.6, color='#3b82f6', label='정상', density=True)
    ax.hist(fraud[col],  bins=50, alpha=0.6, color='#f97316', label='사기', density=True)
    ax.set_title(col, fontsize=9, color='#c9d1d9')
    ax.tick_params(labelsize=7, colors='#8b949e')
    for spine in ax.spines.values(): spine.set_edgecolor('#30363d')
    if ax == axes[0][0]: ax.legend(fontsize=7, facecolor='#161b22', edgecolor='#30363d')

plt.tight_layout()
plt.savefig('../outputs/figures/03_feature_dist.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("\n가장 변별력 높은 피처 Top 10:")
print(diff.head(10).round(3).to_string())

In [ ]:
# ── 시간대별 사기 패턴 ─────────────────────────────────────
df['Hour'] = (df['Time'] % 86400 // 3600).astype(int)

hourly = df.groupby('Hour')['Class'].agg(['sum', 'count', 'mean']).reset_index()
hourly.columns = ['Hour', '사기건수', '총거래', '사기율']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7))
fig.patch.set_facecolor('#0d1117')

for ax in [ax1, ax2]:
    ax.set_facecolor('#161b22')
    for spine in ax.spines.values(): spine.set_edgecolor('#30363d')

ax1.bar(hourly['Hour'], hourly['총거래'], color='#3b82f6', alpha=0.7, label='총 거래')
ax1.set_title('시간대별 거래량', fontsize=11, color='#e6edf3')
ax1.set_ylabel('거래 건수', color='#8b949e')
ax1.tick_params(colors='#8b949e')

ax2.bar(hourly['Hour'], hourly['사기율'] * 100, color='#f97316', alpha=0.8, label='사기율')
ax2.set_title('시간대별 사기율 (%)', fontsize=11, color='#e6edf3')
ax2.set_xlabel('시간 (Hour)', color='#8b949e')
ax2.set_ylabel('사기율 (%)', color='#8b949e')
ax2.tick_params(colors='#8b949e')

plt.tight_layout()
plt.savefig('../outputs/figures/04_hourly_fraud.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

peak_hour = hourly.loc[hourly['사기율'].idxmax(), 'Hour']
print(f"\n사기율 최고 시간대: {peak_hour}시 ({hourly.loc[hourly['사기율'].idxmax(), '사기율']:.4%})")

In [ ]:
# ── 상관관계 히트맵 (Top 피처) ──────────────────────────────
top_feats = diff.head(10).index.tolist() + ['Amount', 'Class']
corr = df[top_feats].corr()

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

import matplotlib.colors as mcolors
cmap = plt.cm.RdBu_r
im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(len(top_feats))); ax.set_yticks(range(len(top_feats)))
ax.set_xticklabels(top_feats, rotation=45, ha='right', fontsize=9, color='#c9d1d9')
ax.set_yticklabels(top_feats, fontsize=9, color='#c9d1d9')

for i in range(len(top_feats)):
    for j in range(len(top_feats)):
        val = corr.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7, color='white' if abs(val) > 0.5 else '#c9d1d9')

plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('피처 상관관계 (Class 포함)', fontsize=12, color='#e6edf3', pad=15)
for spine in ax.spines.values(): spine.set_edgecolor('#30363d')

plt.tight_layout()
plt.savefig('../outputs/figures/05_correlation.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── EDA 요약 및 모델링 전략 ────────────────────────────────
print("="*60)
print("  EDA 핵심 인사이트")
print("="*60)
print("""
1. 클래스 불균형 (0.17%)
   → 평가: ROC-AUC 대신 Average Precision 사용
   → 처리: SMOTE + class_weight 실험

2. V1, V3, V4, V10, V12, V14 피처가 가장 변별력 높음
   → PCA 변환된 거래 패턴 피처
   → Feature selection 고려 가능

3. Amount 분포가 정상/사기 간 차이 있음
   → log1p 변환으로 스케일 조정 필요

4. 시간대별 패턴 존재
   → Hour, IsNight 파생 피처 추가

5. 중복 행 존재 시 제거 필요 (실제 데이터 기준)
""")

print("\n->다음: 02_Modeling.ipynb")